## Sequential Algorithm for Open-Pit Mine Scheduling Problem

### Installing packages

In [ ]:
!pip install -q pandas==3.0.2 numpy==2.4.4 mip==1.17.6 gurobipy==13.0.1 matplotlib==3.10.8 plotly==6.6.0

### Importing packages

In [ ]:
import time, pandas as pd, gurobipy as gp, plotly.express as px, matplotlib.pyplot as plt, numpy as np
from mip import *

### Loading block model file

In [ ]:
# Load block model 
cols = ["id", "x", "y", "z", "tonn", "blockvalue", "destination", "au", "ag", "cu"]

# Read the block model (space-separated format)
blocks = pd.read_csv("p4hd.blocks", sep=r"\s+", header=None, names=cols, engine="python")

# Convert destination=2 into 0 
blocks["destination"] = blocks["destination"].replace({2: 0})

# Add a column to store the extraction period (initially empty)
blocks["period"] = pd.NA

print(f"Loaded {len(blocks):,} blocks")

### Loading block precedences file

In [ ]:
# Load precedence relations (Precedence file format: block_id  n_prec  prec1  prec2  ... )
precedence = {}
with open("p4hd.prec") as f:
    for line in f:
        parts = line.strip().split()
        block_id = int(parts[0])
        n_prec = int(parts[1])
        precedence[block_id] = [int(parts[k]) for k in range(2, 2+n_prec)]

print(f"Loaded {sum(len(precs) for precs in precedence.values()):,} precedences")

### Sequential Algorithm for Open-Pit Mine Scheduling

In [ ]:
# Parameters 
max_period = 6       # Maximum number of scheduling periods
PC = 12_500_000        # Processing capacity (tons)
MC = 52_500_000        # Mine capacity (tons)
alpha = 0.15           # Discount rate for Net Present Value (NPV)

# Sequential Optimization 
code_start = time.time()
period_profit = []   # Store discounted values per period
extracted = set()    # Keep track of already extracted blocks

# Build a model using Gurobi Solver, if available, or use open-source CBC solver (MIP Package Default)
try:
    Model(sense=MAXIMIZE, solver_name="GRB").clear()
    solver_name = "GRB"
except Exception:
    solver_name = "CBC"
print(f"Using {solver_name} solver")
print("Starting optimization")

# Loop over scheduling periods
for t in range(max_period):

    # Available blocks = those not extracted yet
    B = list(blocks.index[~blocks["id"].isin(extracted)])
    if not B:
        break  # Stop if no available blocks remain

    # Extract data for available blocks
    ids = blocks.loc[B, "id"].tolist()
    d = blocks.loc[B, "destination"].tolist()
    v = blocks.loc[B, "blockvalue"].tolist()
    w = blocks.loc[B, "tonn"].tolist()

    model = Model(sense=MAXIMIZE, solver_name=solver_name)
    model.max_seconds = 600  # 10 minutes

    # Decision variables: x[b] = 1 if block b is extracted in this period
    x = [model.add_var(var_type=BINARY) for _ in B]

    # Objective: maximize undiscounted profit in this period
    model.objective = xsum(x[b] * v[b] for b in range(len(B)))

    # Precedence constraints: a block can only be extracted if its predecessors are extracted
    id_to_var = {ids[b]: b for b in range(len(B))}
    for bid in ids:
        if bid in precedence:
            for prec in precedence[bid]:
                if prec in extracted:
                    continue  # Already extracted in earlier period
                elif prec in id_to_var:
                    model += x[id_to_var[bid]] <= x[id_to_var[prec]]
                else:
                    # Predecessor not extracted yet → force this block = 0
                    model += x[id_to_var[bid]] == 0

    # Capacity constraint (only processing blocks where destination=1)
    model += xsum(d[b] * w[b] * x[b] for b in range(len(B))) <= PC
    model += xsum(w[b] * x[b] for b in range(len(B))) <= MC

    # Solve optimization 
    opt_start = time.time()
    model.optimize()
    opt_time = time.time() - opt_start

    # Check solution status
    if model.status in (OptimizationStatus.OPTIMAL, OptimizationStatus.FEASIBLE) and model.objective_value > 0:
        undiscounted_value = model.objective_value
        discounted_value = undiscounted_value / ((1 + alpha) ** t)
        period_profit.append(discounted_value)

        # Update extracted blocks
        files_start = time.time()
        selected = []
        for b, var in enumerate(x):
            if var.x > 0.5:  # If block is selected
                bid = ids[b]
                extracted.add(bid)
                blocks.loc[blocks["id"] == bid, "period"] = t
                selected.append(bid)
        files_time = time.time() - files_start

        # Print summary for this period
        print(f"Period: {t} | "
              f"Undiscounted: {undiscounted_value:,.0f} | " f"Discounted: {discounted_value:,.0f} | " f"Blocks: {len(selected)} | "
              f"Opt Time (min): {opt_time/60:.2f} | " f"Files Time (min): {files_time/60:.2f}")
    else:
        break

    # Clear model (free memory before next period)
    model.clear()

# Final Results 
npv = sum(period_profit)  # Net Present Value
print(f"\nNPV: {npv:,.0f}")
print(f"Total blocks extracted: {len(extracted)}")
print(f"Code Time: {(time.time()-code_start)/60:.2f} minutes")

# Save solution 
blocks.to_csv("p4hd_solution.csv", index=False)

### Mine Scheduling 3D Plot

In [ ]:
# Block dimensions
dx, dy, dz = 50, 50, 20

# Filter extracted blocks and prepare data
d = blocks.loc[blocks.period.notna()].copy()
d["period"] = d["period"].astype(int)
d[["x","y","z"]] *= [dx, dy, dz]
d["period"] -= d["period"].min()
d["period"] = d["period"].astype(str)

# Axis scaling for correct proportions
r = d[["x","y","z"]].max() - d[["x","y","z"]].min()
s = r.max()

# 3D categorical scatter with Plotly default colors
fig = px.scatter_3d(d, x="x", y="y", z="z", color="period", color_discrete_sequence=px.colors.qualitative.Plotly,
    category_orders={"period": sorted(d["period"], key=int)}, labels={"color": "Periods"}, 
    title="Mine Scheduling", width=900, height=500).update_traces(marker=dict(symbol="square", size=7, 
    line=dict(width=0.5, color="black"))).update_layout(scene=dict(xaxis_title="X (m)", yaxis_title="Y (m)", 
    zaxis_title="Elevation (m)", aspectmode="manual", aspectratio=dict(x=r.x/s, y=r.y/s, z=r.z/s)))
fig.show()

In [ ]:
# Salve 3D Plot 
fig.write_html("p4hd_plot.html", full_html=True, include_plotlyjs="cdn",
    config={"responsive": True}, default_width="100vw", default_height="100vh")

### Capacity & Grade Chart

In [ ]:
# Consider only extracted blocks
df = blocks.dropna(subset=["period"]).copy()

# Processed material only (destination == 1)
df["w_proc"]  = df["tonn"] * df["destination"]        # processed tonnage
df["au_proc"] = df["au"] * df["w_proc"]             # contained metal

# Aggregate by period
summary = (df.groupby("period", as_index=False).agg(ton_mine=("tonn", "sum"), ton_plant=("w_proc", "sum"), metal=("au_proc", "sum")))

# Weighted average feed grade
summary["grade_avg"] = summary["metal"] / summary["ton_plant"]
summary["grade_avg"] = summary["grade_avg"].fillna(0)

# Plot
x = np.arange(len(summary))
w = 0.35
fig, ax1 = plt.subplots(figsize=(8, 4))

# Bars: Mine vs Process capacity (Mt)
ax1.bar(x - w/2, summary["ton_mine"] / 1e6, w, label="Mine (Mt)", color="#FFC44D", edgecolor="black")
ax1.bar(x + w/2, summary["ton_plant"] / 1e6, w, label="Process (Mt)", color="#4C5BFF", edgecolor="black")

ax1.set_xlabel("Period")
ax1.set_ylabel("Capacity (Mt)")
ax1.set_xticks(x)
ax1.set_xticklabels(summary["period"].astype(int))

# Y-axis from zero to slightly above max
ax1.set_ylim(0, 1.1 * max(summary["ton_mine"].max(), summary["ton_plant"].max()) / 1e6)

# Secondary axis: Feed grade
ax2 = ax1.twinx()
ax2.plot(x, summary["grade_avg"], color="green", linewidth=2.5, label="Au (oz/ton)")

ax2.set_ylabel("Au (oz/ton)")
ax2.set_ylim(0, 1.15 * summary["grade_avg"].max())

# Visual hierarchy
ax1.patch.set_visible(False)

# Legend
fig.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08), frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# Salve plot
fig.savefig("p4hd_results.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()